In [1]:
# Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



Successfully saved authorization token.


In [1]:
# Initialize Google Earth Engine API.
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm.
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")

def create_monthly_composite(month):
    # Determine the start and end dates for each month
    start_date = ee.Date.fromYMD(2018, month, 1)
    end_date = start_date.advance(1, 'month')
    
    # Filter by date, apply preprocessing, select bands, and compute median
    composite = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                 .filterDate(start_date, end_date)
                 .map(prep_sr_l8)
                 .select(['SR_B4', 'SR_B5'])
                 .median()
                 .set({
                     "system:time_start": start_date.millis(),
                     "month": month,
                     "year": 2018
                 }))
    return composite

# Create an Earth Engine list from 1 to 12
months = ee.List.sequence(1, 12)

# Map the function over the list of months to create a list of images
monthly_images = months.map(create_monthly_composite)

# Cast the resulting list of images back into an ImageCollection
ic = ee.ImageCollection.fromImages(monthly_images)

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
# Code snippet 1: Landsat 8 export of the Red and NIR bands of 
#                 Plumas National Forest, composited for each month of 2018.

# My UDF. This will simply select the Red and NIR bands in my Landsat scene.
def get_bands(df):
    return df[["SR_B4", "SR_B5"]]


from robustraster import run

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": get_bands,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_Tiles",
        "vrt": True,
        "report": True
    }
)


In [ ]:
# Code Snippet 2: Using the VRTs generated from Code Snippet 1, compute the NDVI of all exported tiles.
# Load in VRT file
# Compute NDVI
# Export results to disk

# My UDF. This time, we compute the NDVI derived from the data obtained
# from Code Snippet 1.

def compute_ndvi(df):
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df["ndvi"]

from robustraster import run
import glob as glob

files = glob.glob("./Plumas_Tiles/time_2018*.vrt")

run(
    dataset=files,
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles",
        "report": True
    }
)

In [3]:
# Code Snippet 3: Rather than pulling the Landsat data first with get_bands(), we use our ee.ImageCollection
#                 as our UDD, compute NDVI, and export the results to a parquet file to disk.

def compute_ndvi(df):
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df["ndvi"]

from robustraster import run

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
     function_tuning_config={
        "tune_function": True
    },
    export_config={
        "mode": "vector",
        "file_format": "parquet",
        "output_folder": "Plumas_NDVI_Parquet",
        "report": True
    }
)

Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:64824' processes=16 threads=16, memory=29.59 GiB>
[robustraster] Dask workers authenticated to Earth Engine.


[robustraster] Tuning user function...
Memory Benchmark -> Actual: 0.36 GiB | Ceiling limit: 16.00 GiB
Previous processing time: 3.6970991897845734e-08
Latest processing time: 1.932947888153936e-08
Algorithm Improvement Rate: 47.72%
Difference improved remarkably scaling constraints! (> 5% reduction gains)
Memory Benchmark -> Actual: 0.42 GiB | Ceiling limit: 16.00 GiB
Previous processing time: 1.932947888153936e-08
Latest processing time: 8.7028345931806e-09
Algorithm Improvement Rate: 54.98%
Difference improved remarkably scaling constraints! (> 5% reduction gains)
Memory Benchmark -> Actual: 0.54 GiB | Ceiling limit: 16.00 GiB
Previous processing time: 8.7028345931806e-09
Latest processing time: 4.683638902060074e-09
Algorithm Improvement Rate: 46.18%
Difference improved remarkably scaling constraints! (> 5% reduction gains)
[robustraster] Tuned chunk has exceeded Google Earth Engine's max chunk size!
[robustraster] Setting tuned chunk size to Google Earth Engine's max chunk size...

In [3]:
# Code Snippet 4: Using our ee.ImageCollection as our UDD again, we pass in an R syntaxed version of
#                 our NDVI function. We export the results to Google Cloud Storage.

from robustraster import run

# My UDF. We are still computing NDVI, but using R syntax.
r_code = """\
compute_ndvi_r <- function(df) {
    df$ndvi <- (df$SR_B5 - df$SR_B4) / (df$SR_B5 + df$SR_B4)
    return(df[, c("time", "X", "Y", "ndvi")])
}
"""

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "is_r_function": True,
        "r_function_code": r_code,
        "r_function_name": "compute_ndvi_r",
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles_R_Docker_GCS",
        "export_to_gcs": True,
        "gcs_credentials": r"C:\Users\Adriano Matos\Downloads\admin-493800-d65989e136bf.json",
        "gcs_bucket": "robustraster-bucket",
    },
    #docker_image="adrianomdocker/robustraster"
    docker_image = "adrianomdocker/r042"
)
# #path\to\service-account-credentials.jso

# run(
#     dataset=ic,
#     dataset_config={
#         'geometry': Plumas_Boundaries,
#         'crs': 'EPSG:3310',
#         'scale': 30,
#     },
#     user_function_config={
#         "is_r_function": True,
#         "r_function_code": r_code,
#         "r_function_name": "compute_ndvi_r",
#     },
#     export_config={
#         "mode": "raster",
#         "file_format": "GTiff",
#         "output_folder": "Plumas_NDVI_Tiles_R_Docker"
#     },
#     docker_image = "adrianomdocker/r042"
# )

[robustraster] Found EE credentials at: C:\Users\Adriano Matos/.config/earthengine/credentials
[robustraster] Mounting EE credentials to /root/.config/earthengine/credentials
[robustraster] Mounting output: C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\demos\Plumas_NDVI_Tiles_R_Docker_GCS -> /robustraster_output
[robustraster] Mounting active package source: C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src -> /robustraster_src
Dask Scheduler started: dask-366887-scheduler (8ca1aa172c55)
Dask Worker 1 started: dask-366887-worker-1 (98a16ae13108) mapped to port 61998
Dask Worker 2 started: dask-366887-worker-2 (49df0499fdc0) mapped to port 61999
Dask Worker 3 started: dask-366887-worker-3 (47c2b8730c32) mapped to port 62000
Dask Worker 4 started: dask-366887-worker-4 (f2fbe83f7639) mapped to port 62001
Dask Worker 5 started: dask-366887-worker-5 (8e95010426cc) mapped to port 62002
Dask Worker 6 started: dask-366887-worker-6 (cd9454a1eb31) mapped t

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\distributed\client.py:1617: VersionMismatchWarning: Mismatched versions found

+---------+-----------------+-----------------+-----------------+
| Package | Client          | Scheduler       | Workers         |
+---------+-----------------+-----------------+-----------------+
| numpy   | 2.2.6           | 2.0.1           | 2.0.1           |
| pandas  | 2.3.3           | 2.2.2           | 2.2.2           |
| python  | 3.10.19.final.0 | 3.10.20.final.0 | 3.10.20.final.0 |
| tornado | 6.5.4           | 6.4.2           | 6.4.2           |
+---------+-----------------+-----------------+-----------------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Docker Dask cluster started: <Client: 'tcp://172.20.0.2:8786' processes=12 threads=12, memory=19.31 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] AOI tiling enabled. Streaming tiles in batches...
[robustraster] Processing tile 1 of 15
[robustraster] Running user function...
Bucket robustraster-bucket already exists.
[robustraster] ❌ HttpError during run(): Invalid Credentials, 401
[robustraster] Dask client closed.
[robustraster] Dask cluster shut down.


HttpError: Invalid Credentials, 401